# PhonePe Data Load – Staging Layer

## Objective

This notebook loads raw PhonePe transaction data (CSV) into the MySQL staging table:

`phonepe_transactions_staging`

### Important:
- Python is used ONLY for data loading.
- All business logic (aggregation, YoY, ranking) will be implemented in SQL.
- This maintains a SQL-first Analytics Engineering architecture.

In [1]:
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import os

In [2]:
# Load environment variables from .env file
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

print(f"Database: {DB_NAME}")

Database: llm_governance_fintech


In [3]:
# Update path if needed
file_path = r"C:\PROJECTS\LLM Governance Cost Benchmarking\01_Data\02_Processed_Data\phonepe_transactions_aggregated.csv"

df = pd.read_csv(file_path)

print("Rows loaded from CSV:", len(df))
df.head()

Rows loaded from CSV: 5034


,state,year,quarter,transaction_type,transaction_count,transaction_amount_rupees,transaction_amount_crore,avg_transaction_value,region
0,andaman-&-nicobar-islands,2018,1,Financial Services,33,1.060142e+04,0.001060,321.255149,North-East
1,andaman-&-nicobar-islands,2018,1,Merchant payments,298,4.525072e+05,0.045251,1518.480432,North-East
2,andaman-&-nicobar-islands,2018,1,Others,256,1.846899e+05,0.018469,721.444790,North-East
3,andaman-&-nicobar-islands,2018,1,Peer-to-peer payments,1871,1.213866e+07,1.213866,6487.790112,North-East
4,andaman-&-nicobar-islands,2018,1,Recharge & bill payments,4200,1.845307e+06,0.184531,439.358921,North-East


In [4]:
## What we are doing:
# - Standardizing column names
# - Ensuring correct data types
# - No aggregation
# - No business logic

df.columns = df.columns.str.strip().str.lower()

df["year"] = df["year"].astype(int)
df["quarter"] = df["quarter"].astype(int)
df["transaction_count"] = df["transaction_count"].astype(int)

df["transaction_amount_rupees"] = df["transaction_amount_rupees"].astype(float)
df["transaction_amount_crore"] = df["transaction_amount_crore"].astype(float)
df["avg_transaction_value"] = df["avg_transaction_value"].astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5034 entries, 0 to 5033
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   state                      5034 non-null   object 
 1   year                       5034 non-null   int64  
 2   quarter                    5034 non-null   int64  
 3   transaction_type           5034 non-null   object 
 4   transaction_count          5034 non-null   int64  
 5   transaction_amount_rupees  5034 non-null   float64
 6   transaction_amount_crore   5034 non-null   float64
 7   avg_transaction_value      5034 non-null   float64
 8   region                     5034 non-null   object 
dtypes: float64(3), int64(3), object(3)
memory usage: 354.1+ KB


In [5]:
## What we are doing:
# - Establishing secure connection using environment variables
# - Connecting to llm_governance_fintech database

connection = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

cursor = connection.cursor()

print("✅ Connected to MySQL")

✅ Connected to MySQL


In [6]:
## What we are doing:
# - Clearing staging table before load
# - Ensuring idempotent pipeline behavior

cursor.execute("TRUNCATE TABLE phonepe_transactions_staging;")
connection.commit()

print("Staging table truncated.")

Staging table truncated.


In [7]:
## What we are doing:
# - Bulk inserting raw CSV data into staging layer
# - No transformation applied

insert_query = """
INSERT INTO phonepe_transactions_staging
(
    state,
    year,
    quarter,
    transaction_type,
    transaction_count,
    transaction_amount_rupees,
    transaction_amount_crore,
    avg_transaction_value,
    region
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

data_to_insert = list(df.itertuples(index=False, name=None))

cursor.executemany(insert_query, data_to_insert)
connection.commit()

print("✅ Data inserted into staging table.")

✅ Data inserted into staging table.


In [8]:
cursor.execute("SELECT COUNT(*) FROM phonepe_transactions_staging;")
row_count = cursor.fetchone()[0]

print("Rows in staging table:", row_count)

Rows in staging table: 5034


In [9]:
cursor.close()
connection.close()

print("Connection closed.")

Connection closed.
